# Datamine SURTAC Process Example

This notebook demonstrates how to configure and run the **`surtac`** process wrapper in `dmstudio`.

## Process Description

## SURTAC

SURTAC is a survey measurement reduction process.

This command computes the three-dimensional coordinates of points located by angle and distance measurements from a coordinated survey station. This constitutes the second stage in survey data processing. The user will have created a file containing the survey measurements, prior to running this process, by one of the following processes:

* [AED](<aed.md>)

Editor for file creation and modification.

The AED process may be used to input survey observations from the keyboard, from data written in a field note book.

* [SURLOG](<surlog.md>)

Input of survey data from a character system file.

The SURLOG process is used to input a file of measurements that has been transferred from a digital data recorder.

The user can assign display elements of symbol, color and line style for point and strings of points, according to the descriptive code recorded in the survey.

The output files of coordinated points and string segments will then be used as input to the interactive survey editor, in order to visualize and edit the survey in three-dimensions and update the survey database.

* [SURVIG](<survig.md>)

Interactive graphic survey data editor.

### Input Files:

* **control** (Undefined):
  Input control survey station file. This must contain the numeric fields X, Y and Z. A
  field identifying the station name is required and may be of type numeric or
  alpha-numeric. The default station naming field is STATION of type numeric.
  Required=Yes

* **in** (Undefined):
  Required=Yes

* **attribut** (Undefined):
  Input attribute file. This must contain PTYPE, PTEXT, PSYMBOL, PSYMSZE, PCODE and P.
  May also contain an alpha-numeric string type field.
  Required=Yes

### Output Files:

* **pointou** (Undefined):
  Output point file. This will contain fields PID, X, Y, Z, PSYMBOL, PSYMSZE, P and
  PERIOD (numeric, explicit).
  Required=Yes

* **segou** (Undefined):
  Output string segment file. This will contain fields PID1, PID2, PVALUE, PTYPE, PTEXT,
  PCODE, P and PERIOD.
  Required=Yes

### Fields:

* **stype** (Any : IN, ATTRIBUT):
  Optional key field in ATTRIBUT and IN file for assignment of point and string
  attributes. If this field is not supplied by the user, PTYPE will be used by default and
  must exist in the input file.
  Default=Undefined
  Required=No

* **svalue** (Numeric : IN):
  Optional key field in IN file for assignment of numeric string identifiers to the
  output string file. If this field is not supplied, field PVALUE will be used and must
  exist in the input file.
  Default=Undefined
  Required=No

* **station** (Character : CONTROL):
  Optional alphanumeric survey station identifier in the input CONTROL survey station
  file. If this field is not supplied the default numeric field STATION will be used.
  Default=Undefined
  Required=No

### Parameters:

* **obtype**:
  Input tacheometry observation type. This will be one of: 1 = Horizontal, vertical
  angles and slope distance measurements. 2 = Horizontal angle and distance and vertical
  difference. 3 = Horizontal, vertical angles and staff/stadia intercepts. 4 = Reduced
  X,Y,Z coordinates. The default observation type will be (1).
  Range=1,4
  Values=1,2,3,4
  Default=1
  Required=Yes

* **angle**:
  Angle units used. This will be one of: 1 = Degrees, minutes and seconds; 0-360 2 =
  Gradians; 0-400 The default angle unit will be Degrees, minutes and seconds (1).
  Range=1,2
  Values=1,2
  Default=1
  Required=No

* **voffset**:
  Measurement method for UP/DOWN offsets. This will be one of:- 1 = UP/DOWN offsets
  measured vertically above/below distance measurement. 2 = UP/DOWN offsets measured
  perpendicular to the distance measurement. The default method will be vertical offset
  measurements (1).
  Range=1,2
  Values=1,2
  Default=1
  Required=No

* **period**:
  Numeric integer period number to be stored with output point and string segment data.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **pidstart**:
  Optional start point number for output to point file, field PID, if a numeric TARGET
  field does not exist in the input file. The default start number is 1.(1)
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **pidincr**:
  Optional point number increment, if no TARGET field exists in the input file. The
  default increment is 1.(1)
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **pvstart**:
  Optional start string number for output to string file, field PVALUE, if no PVALUE
  field exists in the input file. The default start number is 1.(1)
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **pvincr**:
  Optional string number increment, if no PVALUE field exists in the input file. The
  default increment is 1.(1)
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **plane**:
  Optional reference plane elevation to which horizontal distances will be reduced prior
  to computation of grid distance and point coordinates. The default is absent data [-],
  no futher reductions are made to the horizontal distance.(-)
  Range=Undefined
  Values=Undefined
  Default=-
  Required=No

* **factor**:
  Optional scale factor to be applied to the plane distance, to compute the grid
  distance, prior to computation of point coordinates. The default value is 1.(1)
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **rocheck**:
  Optional flag = 1 to compare computed base line distance components with check
  measurements made. The default is 0, not to compare check base measurements, if present
  .(0)
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **update**:
  Optional flag = 1 to append computed points to the CONTROL file, if they are
  subsequently occupied by the survey instrument for continuation of the detail survey
  during execution of the process. The default is 0, not to append any computed points to
  the input CONTROL file. (0)
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('surtac')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.initialize_sandbox(notebook_folder, files_to_copy=files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute surtac
print("Running surtac...")
dm_cmd.surtac(
    control_i='t_input_file',  # required
    in_i='t_assays',  # required
    attribut_i='t_SurfaceTriangles',  # required
    pointou_o='t_surtac_out',  # required
    segou_o='t_surtac_out',  # required
    obtype_p='required_val',  # required
    # stype_f='optional',  # optional
    # svalue_f='optional',  # optional
    # station_f='optional',  # optional
    # angle_p=1,  # optional
    # voffset_p=1,  # optional
    # period_p='optional',  # optional
    # pidstart_p=1,  # optional
    # pidincr_p=1,  # optional
    # pvstart_p=1,  # optional
    # pvincr_p=1,  # optional
    # plane_p='-',  # optional
    # factor_p=1,  # optional
    # rocheck_p=0,  # optional
    # update_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("surtac execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_surtac_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")